In [2]:
# =========================================
# 01_bootstrap.ipynb - Cell 1 (SSOT 강제본)
# 목적:
# 1) 프로젝트 루트(QP2_ROOT)를 단일 진실로 '강제'
# 2) .env 로드 (민감 설정 분리)
# 3) 로깅/세션 기본값 세팅
# =========================================

from __future__ import annotations

import os
from pathlib import Path
from datetime import datetime
import logging
import pandas as pd
from dotenv import load_dotenv

# ---- 1) .env 로드 ------------------------------------------------------------
load_dotenv()

# ---- 2) 프로젝트 ROOT 강제 ----------------------------------------------------
# 이 프로젝트에서는 "추정"을 허용하지 않음
env_root = os.getenv("QP2_ROOT", "").strip()
if not env_root:
    raise RuntimeError(
        "QP2_ROOT가 .env에 정의되어 있지 않소.\n"
        "반드시 .env에 다음을 명시하시오:\n"
        "QP2_ROOT=C:/QP2"
    )

QP2_ROOT = Path(env_root).resolve()
EXPECTED_ROOT = Path("C:/QP2").resolve()

if QP2_ROOT != EXPECTED_ROOT:
    raise RuntimeError(
        f"QP2_ROOT 불일치:\n"
        f"  env QP2_ROOT = {QP2_ROOT}\n"
        f"  expected     = {EXPECTED_ROOT}\n"
        f"루트를 바로잡고 다시 실행하시오."
    )

# ---- 3) 표준 디렉토리 (SSOT) --------------------------------------------------
DATA_DIR = QP2_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
META_DIR = DATA_DIR / "meta"

for p in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, META_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ---- 4) 기본 설정값 -----------------------------------------------------------
SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "").strip()
SEC_RATE_LIMIT_PER_SEC = float(os.getenv("SEC_RATE_LIMIT_PER_SEC", "8"))
HTTP_TIMEOUT = int(os.getenv("HTTP_TIMEOUT", "30"))

if not SEC_USER_AGENT:
    raise RuntimeError(
        "SEC_USER_AGENT가 비어있소.\n"
        ".env에 '이름 이메일' 형식으로 넣으시오."
    )

# ---- 5) 로깅 ------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("qp2")

logger.info("QP2_ROOT = %s", QP2_ROOT)
logger.info("DATA_DIR = %s", DATA_DIR)
logger.info(
    "SEC_RATE_LIMIT_PER_SEC=%s | HTTP_TIMEOUT=%s",
    SEC_RATE_LIMIT_PER_SEC,
    HTTP_TIMEOUT,
)

# ---- 6) 실행 스냅샷 -----------------------------------------------------------
snapshot = {
    "run_utc": datetime.utcnow().isoformat(timespec="seconds"),
    "qp2_root": str(QP2_ROOT),
    "sec_rate_limit_per_sec": SEC_RATE_LIMIT_PER_SEC,
    "http_timeout": HTTP_TIMEOUT,
}
pd.Series(snapshot).to_json(META_DIR / "last_run_snapshot.json", force_ascii=False)

# ---- 7) 표준 저장 함수 --------------------------------------------------------
def save_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)

logger.info("Bootstrap complete (SSOT locked).")


2026-01-27 10:43:19,827 | INFO | qp2 | QP2_ROOT = C:\QP2
2026-01-27 10:43:19,828 | INFO | qp2 | DATA_DIR = C:\QP2\data
2026-01-27 10:43:19,829 | INFO | qp2 | SEC_RATE_LIMIT_PER_SEC=8.0 | HTTP_TIMEOUT=30
C:\Users\이노\AppData\Local\Temp\ipykernel_23836\418771218.py:80: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "run_utc": datetime.utcnow().isoformat(timespec="seconds"),
2026-01-27 10:43:19,838 | INFO | qp2 | Bootstrap complete (SSOT locked).


In [3]:
# =========================================
# 01_bootstrap.ipynb - Cell 2 (방탄판)
# - SEC ticker->CIK map 다운로드 + 캐시 + 상세 에러 출력
# =========================================

import requests
import pandas as pd
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

SEC_TICKER_CIK_URL = "https://www.sec.gov/files/company_tickers.json"
OUT_PATH = META_DIR / "sec_ticker_cik.parquet"

# 0) 필수값 점검 (여기서 터지면 .env 문제)
if not SEC_USER_AGENT or not SEC_USER_AGENT.strip():
    raise RuntimeError("SEC_USER_AGENT 비어있소. .env에 '이름 이메일' 형태로 넣으시오.")

session = requests.Session()

@retry(
    reraise=True,
    stop=stop_after_attempt(6),
    wait=wait_exponential(multiplier=1, min=1, max=30),
    retry=retry_if_exception_type((requests.RequestException,)),
)
def fetch_sec_json(url: str) -> dict:
    """
    SEC는 403/429/일시 네트워크 에러가 잦아 재시도 포함.
    실패 시 상태코드/본문 일부까지 출력해 원인 추적 가능하게 함.
    """
    headers = {
        "User-Agent": SEC_USER_AGENT,
        "Accept": "application/json",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
    }
    r = session.get(url, headers=headers, timeout=HTTP_TIMEOUT)

    # 에러면 상세 로그
    if not r.ok:
        body_head = (r.text or "")[:300].replace("\n", " ")
        raise requests.HTTPError(
            f"SEC request failed: status={r.status_code}, body_head={body_head}"
        )

    return r.json()


def download_sec_ticker_cik(force: bool = False) -> pd.DataFrame:
    # 1) 캐시 있으면 재사용
    if OUT_PATH.exists() and not force:
        logger.info("Using cached SEC ticker map: %s", OUT_PATH)
        return pd.read_parquet(OUT_PATH)

    # 2) 다운로드
    logger.info("Downloading SEC ticker->CIK map: %s", SEC_TICKER_CIK_URL)
    raw = fetch_sec_json(SEC_TICKER_CIK_URL)

    # 3) 정규화
    rows = []
    for _, v in raw.items():
        rows.append(
            {
                "ticker": str(v["ticker"]).upper().strip(),
                "cik": str(v["cik_str"]).zfill(10),
                "title": str(v.get("title", "")).strip(),
            }
        )

    df = (
        pd.DataFrame(rows)
        .dropna(subset=["ticker", "cik"])
        .drop_duplicates(subset=["ticker"])
        .sort_values("ticker")
        .reset_index(drop=True)
    )

    # 4) 저장
    save_parquet(df, OUT_PATH)
    logger.info("Saved SEC ticker map (%d rows) -> %s", len(df), OUT_PATH)
    return df


# 실행
sec_map = download_sec_ticker_cik(force=False)
sec_map.head(10)


2026-01-27 10:43:22,843 | INFO | qp2 | Using cached SEC ticker map: C:\QP2\data\meta\sec_ticker_cik.parquet


,ticker,cik,title
0,A,0001090872,"AGILENT TECHNOLOGIES, INC."
1,AA,0001675149,Alcoa Corp
2,AAAU,0001708646,Goldman Sachs Physical Gold ETF
3,AACB,0002034334,Artius II Acquisition Inc.
4,AACBR,0002034334,Artius II Acquisition Inc.
5,AACBU,0002034334,Artius II Acquisition Inc.
6,AACG,0001420529,ATA Creativity Global
7,AACO,0002099906,Abony Acquisition Corp. I
8,AAL,0000006201,American Airlines Group Inc.
9,AAM,0002012964,AA Mission Acquisition Corp.


In [4]:
# =========================================
# 01_bootstrap.ipynb - Cell 3 (Wikipedia 403 우회)
# 목적:
# 1) Wikipedia에서 S&P500 구성 테이블을 "requests + UA"로 가져온다
# 2) Yahoo 표기(점->대시)로 티커 표준화
# 3) SEC ticker->CIK(sec_map)과 결합하여 master_universe 생성
# 4) meta에 저장 + unmatched 저장
# =========================================

import pandas as pd
import requests
from io import StringIO

SP500_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

OUT_UNIVERSE = META_DIR / "sp500_universe.parquet"
OUT_UNMATCHED = META_DIR / "sp500_unmatched.parquet"

def to_yahoo_ticker(t: str) -> str:
    return str(t).upper().strip().replace(".", "-")

def fetch_html(url: str) -> str:
    # Wikipedia 403 우회: 브라우저 흉내
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Connection": "keep-alive",
    }
    r = requests.get(url, headers=headers, timeout=HTTP_TIMEOUT)
    r.raise_for_status()
    return r.text

logger.info("Loading S&P500 table from Wikipedia (UA bypass)")
html = fetch_html(SP500_URL)

# read_html은 문자열을 받으면 파일처럼 읽을 수 있어야 하므로 StringIO로 감싼다
tables = pd.read_html(StringIO(html))
sp = tables[0].copy()
sp.columns = [c.strip() for c in sp.columns]

# 방어적 체크
if "Symbol" not in sp.columns or "Security" not in sp.columns:
    raise RuntimeError(f"Unexpected Wikipedia table columns: {list(sp.columns)}")

# S&P500 쪽 티커 정리
sp["ticker_sp"] = sp["Symbol"].astype(str).str.upper().str.strip()
sp["ticker_yahoo"] = sp["ticker_sp"].map(to_yahoo_ticker)
sp["join_key"] = sp["ticker_yahoo"]

# SEC 쪽도 join_key 생성 (충돌 방지 위해 필요한 컬럼만 남김)
sec_key = sec_map.copy()
sec_key["join_key"] = sec_key["ticker"].map(to_yahoo_ticker)
sec_key = (
    sec_key[["join_key", "cik", "title"]]
    .drop_duplicates(subset=["join_key"])
)

# 매칭
master = sp.merge(sec_key, on="join_key", how="left")
master = master.rename(columns={"Security": "security_name"})

# 최종 컬럼 정리
keep_cols = ["ticker_sp", "ticker_yahoo", "cik", "security_name"]
for c in ["GICS Sector", "GICS Sub-Industry"]:
    if c in master.columns:
        keep_cols.append(c)

master = master[keep_cols].copy()
unmatched = master[master["cik"].isna()].copy()

# 저장
save_parquet(master, OUT_UNIVERSE)
save_parquet(unmatched, OUT_UNMATCHED)

logger.info("Saved universe: %s (rows=%d, match_rate=%.4f)",
            OUT_UNIVERSE, len(master), master["cik"].notna().mean())
logger.info("Unmatched: %s (rows=%d)", OUT_UNMATCHED, len(unmatched))

master.head(10), float(master["cik"].notna().mean()), unmatched.head(10)


2026-01-27 10:43:26,631 | INFO | qp2 | Loading S&P500 table from Wikipedia (UA bypass)
2026-01-27 10:43:27,631 | INFO | qp2 | Saved universe: C:\QP2\data\meta\sp500_universe.parquet (rows=503, match_rate=1.0000)
2026-01-27 10:43:27,632 | INFO | qp2 | Unmatched: C:\QP2\data\meta\sp500_unmatched.parquet (rows=0)


(  ticker_sp ticker_yahoo         cik           security_name  \
 0       MMM          MMM  0000066740                      3M   
 1       AOS          AOS  0000091142             A. O. Smith   
 2       ABT          ABT  0000001800     Abbott Laboratories   
 3      ABBV         ABBV  0001551152                  AbbVie   
 4       ACN          ACN  0001467373               Accenture   
 5      ADBE         ADBE  0000796343              Adobe Inc.   
 6       AMD          AMD  0000002488  Advanced Micro Devices   
 7       AES          AES  0000874761         AES Corporation   
 8       AFL          AFL  0000004977                   Aflac   
 9         A            A  0001090872    Agilent Technologies   
 
               GICS Sector                             GICS Sub-Industry  
 0             Industrials                      Industrial Conglomerates  
 1             Industrials                             Building Products  
 2             Health Care                         Health 

In [5]:
# =========================================
# Cell 4) Yahoo 증분 업데이트 (NY 기준 날짜로 수정: 미래 날짜 요청 차단)
# =========================================

import yfinance as yf
from tqdm import tqdm
import pandas as pd
import time
from pathlib import Path

YAHOO_DIR = RAW_DIR / "yahoo"
YAHOO_DIR.mkdir(parents=True, exist_ok=True)
FAILED_PATH = META_DIR / "yahoo_failed.parquet"

# --- yfinance 에러 로그 소음 줄이기(원하면 주석 처리 가능) ---
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

def _ensure_date_col(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        return df
    if "date" in df.columns:
        df = df.rename(columns={"date": "Date"})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        return df
    if isinstance(df.index, (pd.DatetimeIndex, pd.PeriodIndex)):
        df = df.reset_index()
        first = df.columns[0]
        df = df.rename(columns={first: "Date"})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        return df
    tmp = df.reset_index()
    first = tmp.columns[0]
    tmp = tmp.rename(columns={first: "Date"})
    tmp["Date"] = pd.to_datetime(tmp["Date"], errors="coerce")
    return tmp

def _read_last_date(path: Path) -> pd.Timestamp | None:
    try:
        df = pd.read_parquet(path)
        df = _ensure_date_col(df)
        d = df["Date"].dropna()
        if d.empty:
            return None
        return d.max().normalize()
    except Exception as e:
        logger.warning("Failed to read last_date: %s | %s", path.name, e)
        return None

def _download_range(ticker: str, start: str | None, end: str | None) -> pd.DataFrame | None:
    try:
        df = yf.download(
            ticker,
            start=start,
            end=end,
            progress=False,
            threads=False,
            auto_adjust=False,
        )
        if df is None or df.empty:
            return None
        df = df.reset_index()
        df = _ensure_date_col(df)
        df["ticker"] = ticker
        return df
    except Exception as e:
        logger.warning("Yahoo download failed: %s | %s", ticker, e)
        return None

def _merge_and_save(path: Path, df_add: pd.DataFrame) -> None:
    if path.exists():
        df_old = pd.read_parquet(path)
        df_old = _ensure_date_col(df_old)
        combined = pd.concat([df_old, df_add], ignore_index=True)
    else:
        combined = df_add.copy()

    combined = _ensure_date_col(combined)
    combined = combined.dropna(subset=["Date"])
    combined = combined.sort_values("Date").drop_duplicates(subset=["Date"], keep="last")

    if "ticker" not in combined.columns:
        combined["ticker"] = path.stem

    save_parquet(combined, path)

# --- ✅ 핵심: "미국 장(뉴욕) 기준" 오늘 ---
market_today = pd.Timestamp.now(tz="America/New_York").normalize().tz_localize(None)
end_str = (market_today + pd.Timedelta(days=1)).strftime("%Y-%m-%d")  # 내일까지만 허용

failed = []
logger.info("Start Yahoo incremental refresh (NY today=%s) (%d tickers)", market_today.date(), len(master))

for _, row in tqdm(master.iterrows(), total=len(master)):
    t = row["ticker_yahoo"]
    out = YAHOO_DIR / f"{t}.parquet"

    last_date = _read_last_date(out) if out.exists() else None

    if last_date is None:
        # 최초/비정상: 전체 다운로드 (end는 NY 기준)
        df_new = _download_range(t, start="1900-01-01", end=end_str)
        if df_new is None:
            failed.append({"ticker": t, "reason": "initial_or_redownload_empty"})
            continue
        try:
            _merge_and_save(out, df_new)
        except Exception as e:
            failed.append({"ticker": t, "reason": f"save_failed: {e}"})
        time.sleep(0.2)
        continue

    start_date = last_date + pd.Timedelta(days=1)

    # ✅ 미래 날짜 요청 차단: NY 기준 오늘보다 크면 스킵
    if start_date > market_today:
        continue

    start_str = start_date.strftime("%Y-%m-%d")
    df_add = _download_range(t, start=start_str, end=end_str)

    if df_add is None:
        # 장이 아직 안 열렸거나(시간대) 휴일이면 비어있을 수 있음: 실패로 보지 말고 기록만
        failed.append({"ticker": t, "reason": "no_new_rows_or_market_not_ready", "start": start_str})
        continue

    try:
        _merge_and_save(out, df_add)
    except Exception as e:
        failed.append({"ticker": t, "reason": f"merge_failed: {e}", "start": start_str})

    time.sleep(0.2)

failed_df = pd.DataFrame(failed)
save_parquet(failed_df, FAILED_PATH)
logger.info("Yahoo refresh done. Logged=%d | %s", len(failed_df), FAILED_PATH)

failed_df.head(20)


2026-01-27 10:43:29,805 | INFO | qp2 | Start Yahoo incremental refresh (NY today=2026-01-26) (503 tickers)
100%|██████████| 503/503 [00:04<00:00, 111.05it/s]
2026-01-27 10:43:34,344 | INFO | qp2 | Yahoo refresh done. Logged=0 | C:\QP2\data\meta\yahoo_failed.parquet


""


In [6]:
# =========================================
# Cell 4.5) Yahoo RAW 정화(클린 포맷 강제 덮어쓰기) - 멀티컬럼/Date 혼선 완전 대응
# 목적:
# 1) yfinance 결과가 멀티컬럼으로 와도 강제 평탄화
# 2) Date 컬럼이 ('Date','') 등으로 와도 강제 탐지/통일
# 3) raw/yahoo/{ticker}.parquet를 "표준 단일 컬럼"으로 덮어써 오염 제거
# 산출물:
# - data/raw/yahoo/{ticker}.parquet (클린 포맷)
# - data/meta/yahoo_clean_rebuild_failed.parquet
# =========================================

import yfinance as yf
from tqdm import tqdm
import pandas as pd
import time
from pathlib import Path

YAHOO_DIR = RAW_DIR / "yahoo"
YAHOO_DIR.mkdir(parents=True, exist_ok=True)

FAILED_REBUILD_PATH = META_DIR / "yahoo_clean_rebuild_failed.parquet"

def _flatten_cols(cols) -> list[str]:
    out = []
    for c in cols:
        if isinstance(c, tuple):
            out.append(str(c[0]).strip())
        else:
            out.append(str(c).strip())
    # 중복 제거(같은 이름 여러개면 첫 번째만)
    # (pandas에서 df.loc[:, ~duplicated]로 처리할 예정)
    return out

def _ensure_date_col(df: pd.DataFrame) -> pd.DataFrame:
    """
    멀티컬럼/인덱스 어떤 형태든 최종적으로 'Date' 컬럼을 보장한다.
    """
    df = df.copy()

    # 1) 컬럼 평탄화 (멀티컬럼이면 ('Adj Close','AAPL') -> 'Adj Close')
    df.columns = _flatten_cols(df.columns)
    df = df.loc[:, ~pd.Index(df.columns).duplicated(keep="first")]

    # 2) Date 후보 찾기 (대소문자 무시)
    lower_map = {c.lower(): c for c in df.columns}
    if "date" in lower_map:
        real = lower_map["date"]
        if real != "Date":
            df = df.rename(columns={real: "Date"})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        return df

    # 3) 없으면 인덱스가 날짜인지 확인
    if isinstance(df.index, (pd.DatetimeIndex, pd.PeriodIndex)):
        df = df.reset_index()
        df.columns = _flatten_cols(df.columns)
        df = df.loc[:, ~pd.Index(df.columns).duplicated(keep="first")]
        # reset_index 후 다시 date 탐지
        lower_map = {c.lower(): c for c in df.columns}
        if "date" in lower_map:
            real = lower_map["date"]
            if real != "Date":
                df = df.rename(columns={real: "Date"})
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            return df

    # 4) 최후: 첫 컬럼을 Date로 간주
    df = df.reset_index()
    df.columns = _flatten_cols(df.columns)
    df = df.loc[:, ~pd.Index(df.columns).duplicated(keep="first")]
    first = df.columns[0]
    df = df.rename(columns={first: "Date"})
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df


def _download_clean(ticker: str, start: str = "1900-01-01") -> pd.DataFrame | None:
    try:
        df = yf.download(
            ticker,
            start=start,
            progress=False,
            threads=False,
            auto_adjust=False,
            group_by="column",   # 멀티컬럼 가능성 낮추려는 시도(그래도 방어함)
        )
        if df is None or df.empty:
            return None

        df = df.reset_index()
        df = _ensure_date_col(df)

        df = df.dropna(subset=["Date"]).sort_values("Date")
        df["ticker"] = ticker

        # 표준 컬럼만 남기기
        keep = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "ticker"]
        cols = [c for c in keep if c in df.columns]
        df = df[cols].copy()

        df = df.drop_duplicates(subset=["Date"], keep="last").reset_index(drop=True)
        return df

    except Exception as e:
        logger.warning("Yahoo clean download failed: %s | %s", ticker, e)
        return None


failed = []
logger.info("[Cell4.5] Yahoo RAW clean rebuild start (%d tickers)", len(master))

for _, row in tqdm(master.iterrows(), total=len(master)):
    t = row["ticker_yahoo"]
    out = YAHOO_DIR / f"{t}.parquet"

    df = _download_clean(t, start="1900-01-01")
    if df is None:
        failed.append({"ticker": t, "reason": "download_empty_or_failed"})
        continue

    save_parquet(df, out)
    time.sleep(0.2)

failed_df = pd.DataFrame(failed)
save_parquet(failed_df, FAILED_REBUILD_PATH)

logger.info("[Cell4.5] Yahoo RAW clean rebuild done. Failed=%d | %s", len(failed_df), FAILED_REBUILD_PATH)
failed_df.head(20)


2026-01-27 10:43:46,706 | INFO | qp2 | [Cell4.5] Yahoo RAW clean rebuild start (503 tickers)
100%|██████████| 503/503 [08:34<00:00,  1.02s/it]
2026-01-27 10:52:21,443 | INFO | qp2 | [Cell4.5] Yahoo RAW clean rebuild done. Failed=0 | C:\QP2\data\meta\yahoo_clean_rebuild_failed.parquet


""


In [7]:
# =========================================
# Cell 5) Yahoo 가격 패널 생성 (증분환경 + parquet 수복 포함, 확정)
# 목적:
# 1) data/raw/yahoo/*.parquet 전체 재로딩
# 2) pyarrow 스키마 충돌(중복 Date 등) 발생 시 자동 수복 후 재시도
# 3) long / wide 패널을 "현재 시점 스냅샷"으로 재생성
# 산출물:
# - data/interim/yahoo_prices_long.parquet
# - data/interim/yahoo_adjclose_wide.parquet
# - (수복 로그) data/meta/yahoo_repair_log.parquet
# 주의:
# - 업데이트 셀이 아님 (항상 전체 재계산)
# - 문제가 있는 원시 parquet는 이 셀에서 "클린 포맷"으로 재저장됨(의도된 동작)
# =========================================

from pathlib import Path
import pandas as pd
import ast

# pyarrow는 여기서 필수
import pyarrow.parquet as pq
import pyarrow as pa

YAHOO_DIR = RAW_DIR / "yahoo"
OUT_LONG = INTERIM_DIR / "yahoo_prices_long.parquet"
OUT_WIDE = INTERIM_DIR / "yahoo_adjclose_wide.parquet"
REPAIR_LOG_PATH = META_DIR / "yahoo_repair_log.parquet"

assert YAHOO_DIR.exists(), f"YAHOO_DIR not found: {YAHOO_DIR}"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

paths = sorted(YAHOO_DIR.glob("*.parquet"))
if len(paths) == 0:
    raise FileNotFoundError(f"No parquet files in {YAHOO_DIR}")

logger.info("[Cell5] Yahoo parquet files found: %d", len(paths))


def _flatten_one(col):
    if isinstance(col, tuple):
        return str(col[0]).strip()

    s = str(col).strip()
    if s.startswith("(") and s.endswith(")"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, tuple) and len(v) > 0:
                return str(v[0]).strip()
        except Exception:
            pass
    return s


def _read_parquet_any(path: Path) -> pd.DataFrame:
    """
    pandas.read_parquet이 Arrow 스키마 충돌로 터지는 케이스를 피하기 위해
    pyarrow로 직접 읽는다.
    """
    table = pq.read_table(path)  # 여기서도 터지면 진짜 파일이 깨진 것
    return table.to_pandas()


def _extract_date_and_price(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    # 컬럼 평탄화
    df = df.copy()
    df.columns = [_flatten_one(c) for c in df.columns]

    # Date 컬럼 후보들(중복 포함)을 모두 수집
    date_candidates = [c for c in df.columns if str(c).lower() == "date"]
    if len(date_candidates) == 0:
        # index로 날짜가 들어온 경우도 대응
        tmp = df.reset_index()
        tmp.columns = [_flatten_one(c) for c in tmp.columns]
        date_candidates = [c for c in tmp.columns if str(c).lower() == "date"]
        df = tmp

    if len(date_candidates) == 0:
        # 최후: 첫 컬럼을 날짜로 가정
        date_col = df.columns[0]
    else:
        date_col = date_candidates[0]  # 중복이면 첫 번째만 쓴다

    date = pd.to_datetime(df[date_col], errors="coerce")

    # price
    # 'Adj Close' 우선, 없으면 'Close'
    if "Adj Close" in df.columns:
        price = df["Adj Close"]
    elif "Close" in df.columns:
        price = df["Close"]
    else:
        raise KeyError(f"{ticker}: no Adj Close/Close. cols_head={list(df.columns)[:10]}")

    out = pd.DataFrame(
        {
            "date": date,
            "ticker": ticker,
            "adj_close": pd.to_numeric(price, errors="coerce"),
        }
    )

    out = (
        out.dropna(subset=["date"])
           .sort_values("date")
           .drop_duplicates(subset=["date"], keep="last")
           .reset_index(drop=True)
    )
    return out


def _repair_parquet_inplace(path: Path, ticker: str) -> dict:
    """
    깨진/중복 컬럼 parquet를 '표준 포맷'으로 재저장한다.
    표준 포맷: Date, Open, High, Low, Close, Adj Close, Volume, ticker (가능한 것만)
    """
    info = {"ticker": ticker, "file": str(path), "repaired": False, "error": None}

    try:
        df = _read_parquet_any(path)
        df.columns = [_flatten_one(c) for c in df.columns]

        # Date 컬럼 선택(중복이면 첫 번째)
        date_candidates = [c for c in df.columns if str(c).lower() == "date"]
        if len(date_candidates) == 0:
            tmp = df.reset_index()
            tmp.columns = [_flatten_one(c) for c in tmp.columns]
            date_candidates = [c for c in tmp.columns if str(c).lower() == "date"]
            df = tmp

        if len(date_candidates) == 0:
            # 최후: 첫 컬럼을 Date로 간주
            df = df.rename(columns={df.columns[0]: "Date"})
        else:
            df = df.rename(columns={date_candidates[0]: "Date"})

        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date")

        # 중복 컬럼 제거(이름이 같은 컬럼이 여러 개면 첫 번째만)
        df = df.loc[:, ~pd.Index(df.columns).duplicated(keep="first")]

        # 표준 컬럼만 남기기(있으면)
        keep = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "ticker"]
        cols = [c for c in keep if c in df.columns]

        # ticker 없으면 추가
        if "ticker" not in df.columns:
            df["ticker"] = ticker
            cols.append("ticker")

        df = df[cols].copy()

        # 저장(기존 파일 덮어씀) — pandas로 쓰면 이제 정상 포맷으로 고정됨
        save_parquet(df, path)
        info["repaired"] = True
        return info

    except Exception as e:
        info["error"] = str(e)
        return info


chunks = []
repair_logs = []
bad_files = []

for p in paths:
    ticker = p.stem

    # 1차 로딩 시도
    try:
        df_raw = pd.read_parquet(p)  # 빠른 경로
        chunks.append(_extract_date_and_price(df_raw, ticker))
        continue
    except Exception as e1:
        # pandas 경로 실패 -> 수복 시도
        logger.warning("[Cell5] Load failed, try repair: %s | %s", p.name, e1)

    rep = _repair_parquet_inplace(p, ticker)
    repair_logs.append(rep)

    if not rep["repaired"]:
        logger.warning("[Cell5] Repair failed: %s | %s", p.name, rep["error"])
        bad_files.append({"ticker": ticker, "error": rep["error"]})
        continue

    # 수복 후 재로딩
    try:
        df_raw = pd.read_parquet(p)
        chunks.append(_extract_date_and_price(df_raw, ticker))
    except Exception as e2:
        logger.warning("[Cell5] Still failed after repair: %s | %s", p.name, e2)
        bad_files.append({"ticker": ticker, "error": str(e2)})

if repair_logs:
    save_parquet(pd.DataFrame(repair_logs), REPAIR_LOG_PATH)
    logger.info("[Cell5] Repair log saved: %s", REPAIR_LOG_PATH)

if not chunks:
    raise RuntimeError("All Yahoo parquet files failed to load even after repair.")

prices_long = (
    pd.concat(chunks, ignore_index=True)
      .drop_duplicates(subset=["ticker", "date"], keep="last")
      .sort_values(["ticker", "date"])
      .reset_index(drop=True)
)

prices_wide = (
    prices_long
    .pivot(index="date", columns="ticker", values="adj_close")
    .sort_index()
)

logger.info(
    "[Cell5] Long rows=%d | tickers=%d | range=%s ~ %s",
    len(prices_long),
    prices_long["ticker"].nunique(),
    prices_long["date"].min().date(),
    prices_long["date"].max().date(),
)
logger.info("[Cell5] Wide missing rate=%.4f", prices_wide.isna().mean().mean())

save_parquet(prices_long, OUT_LONG)
save_parquet(prices_wide.reset_index(), OUT_WIDE)

logger.info("[Cell5] Saved long: %s", OUT_LONG)
logger.info("[Cell5] Saved wide: %s", OUT_WIDE)

if bad_files:
    bad_df = pd.DataFrame(bad_files)
    save_parquet(bad_df, META_DIR / "yahoo_panel_bad_files.parquet")
    logger.warning("[Cell5] Bad files logged: %d", len(bad_df))

display(prices_long.head())
display(prices_wide.head())


2026-01-27 11:00:13,604 | INFO | qp2 | [Cell5] Yahoo parquet files found: 503
2026-01-27 11:00:26,071 | INFO | qp2 | [Cell5] Long rows=4341978 | tickers=503 | range=1962-01-02 ~ 2026-01-26
2026-01-27 11:00:26,085 | INFO | qp2 | [Cell5] Wide missing rate=0.4646
2026-01-27 11:00:27,160 | INFO | qp2 | [Cell5] Saved long: C:\QP2\data\interim\yahoo_prices_long.parquet
2026-01-27 11:00:27,161 | INFO | qp2 | [Cell5] Saved wide: C:\QP2\data\interim\yahoo_adjclose_wide.parquet


,date,ticker,adj_close
0,1999-11-18,A,26.300022
1,1999-11-19,A,24.133257
2,1999-11-22,A,26.300022
3,1999-11-23,A,23.909113
4,1999-11-24,A,24.544207


ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
date,,,,,,,,,,,,,,,,,,,,,
1962-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.089345,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.090672,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.090893,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.088903,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.088682,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# =========================================
# Cell 6) SEC EDGAR companyfacts 업데이트 (N일 캐시 + 304 시도, 확정)
# 목적:
# 1) S&P500 CIK별 companyfacts JSON을 raw로 저장
# 2) ETag/Last-Modified 있으면 304 조건부 GET 시도
# 3) 없더라도 last_fetched_utc 기반 N일 캐시로 재다운을 제어
# 산출물:
# - data/raw/sec/companyfacts/{cik}.json.gz
# - data/meta/sec_companyfacts_manifest.parquet
# - data/meta/sec_companyfacts_failed.parquet
# =========================================

import gzip
import json
import time
import requests
import pandas as pd
from datetime import datetime, timezone

SEC_RAW_DIR = RAW_DIR / "sec"
COMPANYFACTS_DIR = SEC_RAW_DIR / "companyfacts"
COMPANYFACTS_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = META_DIR / "sec_companyfacts_manifest.parquet"
FAIL_PATH = META_DIR / "sec_companyfacts_failed.parquet"

# ---- 정책: 최근 N일 이내면 스킵 ----
CACHE_DAYS = int(os.getenv("SEC_CACHE_DAYS", "7"))  # .env로 조정 가능, 기본 7일
now_utc = datetime.now(timezone.utc)

def _load_manifest() -> pd.DataFrame:
    if MANIFEST_PATH.exists():
        m = pd.read_parquet(MANIFEST_PATH)
    else:
        m = pd.DataFrame(columns=["cik", "etag", "last_modified", "last_fetched_utc", "status"])

    # 방어적으로 컬럼 보강
    for c in ["cik", "etag", "last_modified", "last_fetched_utc", "status"]:
        if c not in m.columns:
            m[c] = None

    m["cik"] = m["cik"].astype(str).str.zfill(10)
    return m.drop_duplicates(subset=["cik"], keep="last").reset_index(drop=True)

manifest = _load_manifest()
manifest_idx = {row["cik"]: i for i, row in manifest.iterrows()}

session = requests.Session()

def _sec_headers(etag: str | None = None, last_modified: str | None = None) -> dict:
    h = {
        "User-Agent": SEC_USER_AGENT,
        "Accept": "application/json",
        "Accept-Encoding": "gzip, deflate",
        "Connection": "keep-alive",
    }
    if etag:
        h["If-None-Match"] = etag
    if last_modified:
        h["If-Modified-Since"] = last_modified
    return h

def _should_skip(cik: str) -> bool:
    if cik not in manifest_idx:
        return False
    lf = manifest.loc[manifest_idx[cik], "last_fetched_utc"]
    if not lf or str(lf).strip() == "" or str(lf) == "nan":
        return False
    try:
        # lf는 '2026-01-27T01:13:35' 같은 형식
        lf_dt = datetime.fromisoformat(str(lf)).replace(tzinfo=timezone.utc)
        age_days = (now_utc - lf_dt).total_seconds() / 86400.0
        return age_days < CACHE_DAYS
    except Exception:
        return False

def save_json_gz(obj: dict, path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def _update_manifest(cik: str, etag: str | None, last_modified: str | None, status: str):
    row = {
        "cik": cik,
        "etag": etag,
        "last_modified": last_modified,
        "last_fetched_utc": datetime.now(timezone.utc).replace(tzinfo=None).isoformat(timespec="seconds"),
        "status": status,
    }
    if cik in manifest_idx:
        manifest.loc[manifest_idx[cik], :] = row
    else:
        manifest.loc[len(manifest)] = row
        manifest_idx[cik] = len(manifest) - 1

# 실행 대상
ciks = master["cik"].dropna().astype(str).str.zfill(10).unique().tolist()
logger.info("SEC companyfacts refresh start. CIKs=%d | CACHE_DAYS=%d", len(ciks), CACHE_DAYS)

sleep_sec = max(0.15, 1.0 / max(1.0, SEC_RATE_LIMIT_PER_SEC))
fail = []

for cik in tqdm(ciks, total=len(ciks)):
    out_path = COMPANYFACTS_DIR / f"{cik}.json.gz"

    # N일 캐시 스킵
    if _should_skip(cik) and out_path.exists():
        _update_manifest(
            cik,
            etag=manifest.loc[manifest_idx[cik], "etag"] if cik in manifest_idx else None,
            last_modified=manifest.loc[manifest_idx[cik], "last_modified"] if cik in manifest_idx else None,
            status=f"skipped_cache_{CACHE_DAYS}d",
        )
        continue

    prev_etag = manifest.loc[manifest_idx[cik], "etag"] if cik in manifest_idx else None
    prev_lm = manifest.loc[manifest_idx[cik], "last_modified"] if cik in manifest_idx else None

    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

    try:
        r = session.get(url, headers=_sec_headers(prev_etag, prev_lm), timeout=HTTP_TIMEOUT)

        if r.status_code == 304:
            _update_manifest(cik, prev_etag, prev_lm, "not_modified_304")
            time.sleep(sleep_sec)
            continue

        if not r.ok:
            body_head = (r.text or "")[:300].replace("\n", " ")
            raise requests.HTTPError(f"status={r.status_code}, body_head={body_head}")

        obj = r.json()
        save_json_gz(obj, out_path)

        new_etag = r.headers.get("ETag")
        new_lm = r.headers.get("Last-Modified")
        _update_manifest(cik, new_etag, new_lm, "downloaded_200")

    except Exception as e:
        logger.warning("companyfacts failed: %s | %s", cik, e)
        fail.append({"cik": cik, "error": str(e)})
        _update_manifest(cik, prev_etag, prev_lm, f"failed: {type(e).__name__}")

    time.sleep(sleep_sec)

save_parquet(manifest, MANIFEST_PATH)
fail_df = pd.DataFrame(fail)
save_parquet(fail_df, FAIL_PATH)

logger.info("SEC companyfacts refresh done. fail=%d | manifest=%s", len(fail_df), MANIFEST_PATH)
manifest.head(10), fail_df.head(10)


2026-01-27 11:00:37,806 | INFO | qp2 | SEC companyfacts refresh start. CIKs=500 | CACHE_DAYS=7
100%|██████████| 500/500 [00:01<00:00, 446.76it/s]
2026-01-27 11:00:38,935 | INFO | qp2 | SEC companyfacts refresh done. fail=0 | manifest=C:\QP2\data\meta\sec_companyfacts_manifest.parquet


(          cik etag last_modified     last_fetched_utc            status
 0  0000066740  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 1  0000091142  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 2  0000001800  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 3  0001551152  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 4  0001467373  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 5  0000796343  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 6  0000002488  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 7  0000874761  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 8  0000004977  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d
 9  0001090872  NaN           NaN  2026-01-27T02:00:37  skipped_cache_7d,
 Empty DataFrame
 Columns: []
 Index: [])

In [11]:
# =========================================
# Cell 7) companyfacts -> fundamentals_annual (경고 제거 확정판: 스키마 통일)
# =========================================

import gzip
import json
import pandas as pd
from pathlib import Path

OUT_FUND_ANNUAL = INTERIM_DIR / "fundamentals_annual.parquet"

# ---- 표준 계정 정의(묶음) ----
STD_MAP = {
    "Assets": ["Assets"],
    "StockholdersEquity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    ],
    "NetIncomeLoss": ["NetIncomeLoss", "ProfitLoss"],
    "CFO": ["NetCashProvidedByUsedInOperatingActivities"],
    "CapEx": ["CapitalExpenditures"],

    "Revenue": [
        "Revenues",
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "SalesRevenueGoodsNet",
        "SalesRevenueServicesNet",
        "RevenuesNetOfInterestExpense",
    ],

    "Liabilities": ["Liabilities"],
}
LIAB_FALLBACK_SUM = ("LiabilitiesCurrent", "LiabilitiesNoncurrent")

_EMPTY_SERIES = pd.DataFrame(columns=["fy", "end", "filed", "unit", "value"])

def load_json_gz(path: Path) -> dict:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)

def _get_fy_series(gaap: dict, concept: str) -> pd.DataFrame:
    node = gaap.get(concept)
    if not node:
        return _EMPTY_SERIES.copy()

    rows = []
    units = node.get("units", {})
    for unit, items in units.items():
        if not isinstance(items, list):
            continue
        for it in items:
            if str(it.get("fp", "")).upper() != "FY":
                continue
            fy = it.get("fy")
            end = it.get("end")
            filed = it.get("filed")
            val = it.get("val")
            if fy is None or end is None or val is None:
                continue
            rows.append({
                "fy": int(fy),
                "end": pd.to_datetime(end, errors="coerce"),
                "filed": pd.to_datetime(filed, errors="coerce"),
                "unit": str(unit),
                "value": pd.to_numeric(val, errors="coerce"),
            })

    df = pd.DataFrame(rows) if rows else _EMPTY_SERIES.copy()
    df = df.dropna(subset=["fy", "end", "value"])
    if df.empty:
        return _EMPTY_SERIES.copy()

    # 같은 fy 다중 관측치면 filed 최신값 우선
    df = df.sort_values(["fy", "filed"]).drop_duplicates(subset=["fy"], keep="last")
    return df

def _pick_best_unit(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return _EMPTY_SERIES.copy()
    df = df.copy()
    if df["unit"].str.contains("USD", case=False, na=False).any():
        df = df[df["unit"].str.contains("USD", case=False, na=False)].copy()
        df = df.sort_values(["fy", "filed"]).drop_duplicates(subset=["fy"], keep="last")
    return df if not df.empty else _EMPTY_SERIES.copy()

def _series_to_value(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    시리즈 DF -> (fy, end, filed, <name>) 로 변환
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=["fy", "end", "filed", name])
    out = df[["fy", "end", "filed", "value"]].rename(columns={"value": name}).copy()
    return out

def _outer_merge_on_fy(base: pd.DataFrame, add: pd.DataFrame) -> pd.DataFrame:
    """
    fy 기준 outer merge.
    end/filed는 가능한 범위에서 '최신'을 대표로 유지.
    """
    if base is None or base.empty:
        return add.copy()

    m = base.merge(add, on="fy", how="outer", suffixes=("", "_add"))

    # end/filed 대표값 갱신: add가 있으면 더 최신(end/filed max)
    for col in ["end", "filed"]:
        if col in m.columns and f"{col}_add" in m.columns:
            m[col] = pd.to_datetime(m[col], errors="coerce")
            m[f"{col}_add"] = pd.to_datetime(m[f"{col}_add"], errors="coerce")
            m[col] = m[[col, f"{col}_add"]].max(axis=1)
            m = m.drop(columns=[f"{col}_add"])

    return m

def _liabilities_fallback(gaap: dict) -> pd.DataFrame:
    """
    Liabilities가 없을 때:
    - Current/Noncurrent 중 하나라도 있으면 합산
    - end/filed는 존재하는 쪽의 값을 가져오고, 둘 다 있으면 max
    """
    cur = _pick_best_unit(_get_fy_series(gaap, LIAB_FALLBACK_SUM[0]))
    non = _pick_best_unit(_get_fy_series(gaap, LIAB_FALLBACK_SUM[1]))

    if cur.empty and non.empty:
        return _EMPTY_SERIES.copy()

    # 공통 스키마로 fy 기준 outer merge
    cur_v = _series_to_value(cur, "cur")
    non_v = _series_to_value(non, "non")

    m = cur_v.merge(non_v, on="fy", how="outer", suffixes=("", "_x"))

    # end/filed 정리: 둘 중 존재하는 것, 둘 다면 최신
    for col in ["end", "filed"]:
        if col not in m.columns:
            m[col] = pd.NaT
        if f"{col}_x" in m.columns:
            m[col] = pd.to_datetime(m[col], errors="coerce")
            m[f"{col}_x"] = pd.to_datetime(m[f"{col}_x"], errors="coerce")
            m[col] = m[[col, f"{col}_x"]].max(axis=1)
            m = m.drop(columns=[f"{col}_x"])

    m["cur"] = pd.to_numeric(m.get("cur"), errors="coerce").fillna(0.0)
    m["non"] = pd.to_numeric(m.get("non"), errors="coerce").fillna(0.0)
    m["value"] = m["cur"] + m["non"]
    m["unit"] = "USD"

    out = m[["fy", "end", "filed", "unit", "value"]].copy()
    out = out.dropna(subset=["fy"])
    out["fy"] = out["fy"].astype(int)
    out = out.sort_values(["fy", "filed"]).drop_duplicates(subset=["fy"], keep="last")
    return out if not out.empty else _EMPTY_SERIES.copy()

def extract_fundamentals_annual(obj: dict, cik: str) -> pd.DataFrame:
    gaap = obj.get("facts", {}).get("us-gaap", {})

    # 1) 표준 계정 시리즈 확보
    series = {}
    for std_name, candidates in STD_MAP.items():
        best = _EMPTY_SERIES.copy()
        for c in candidates:
            df = _pick_best_unit(_get_fy_series(gaap, c))
            if not df.empty:
                best = df
                break
        series[std_name] = best

    # 2) Liabilities fallback
    if series["Liabilities"].empty:
        series["Liabilities"] = _liabilities_fallback(gaap)

    # 3) wide 병합(안정적으로)
    base = None
    for std_name in ["Assets","Liabilities","CFO","NetIncomeLoss","Revenue","StockholdersEquity","CapEx"]:
        df = series.get(std_name, _EMPTY_SERIES.copy())
        add = _series_to_value(df, std_name)
        base = _outer_merge_on_fy(base, add)

    if base is None or base.empty:
        return pd.DataFrame(columns=["cik","fy","Assets","Liabilities","CFO","NetIncomeLoss","Revenue","StockholdersEquity","CapEx"])

    base["cik"] = cik

    cols = ["cik","fy","Assets","Liabilities","CFO","NetIncomeLoss","Revenue","StockholdersEquity","CapEx"]
    for c in cols:
        if c not in base.columns:
            base[c] = pd.NA

    base = base[cols].sort_values(["cik","fy"]).reset_index(drop=True)
    return base

# ---- 실행 ----
paths = sorted((RAW_DIR / "sec" / "companyfacts").glob("*.json.gz"))
if len(paths) == 0:
    raise FileNotFoundError("No companyfacts .json.gz files found.")

logger.info("Normalize fundamentals annual (schema-stable): files=%d", len(paths))

bad = []
chunks = []
for p in tqdm(paths, total=len(paths)):
    cik = p.stem.replace(".json", "").zfill(10)
    try:
        obj = load_json_gz(p)
        chunks.append(extract_fundamentals_annual(obj, cik=cik))
    except Exception as e:
        bad.append({"file": p.name, "error": str(e)})
        logger.warning("Normalize failed: %s | %s", p.name, e)

fund_wide = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
if fund_wide.empty:
    raise RuntimeError("fundamentals_annual is empty.")

fund_wide = fund_wide.merge(
    master[["cik", "ticker_yahoo", "security_name"]].drop_duplicates(),
    on="cik",
    how="left",
)

save_parquet(fund_wide, OUT_FUND_ANNUAL)

bad_df = pd.DataFrame(bad)
logger.info("Saved fundamentals annual: %s | rows=%d | bad_files=%d", OUT_FUND_ANNUAL, len(fund_wide), len(bad_df))
display(fund_wide.head(30))
display(bad_df.head(30))


2026-01-27 11:06:53,279 | INFO | qp2 | Normalize fundamentals annual (schema-stable): files=500
100%|██████████| 500/500 [02:30<00:00,  3.33it/s]
2026-01-27 11:09:23,380 | INFO | qp2 | Saved fundamentals annual: C:\QP2\data\interim\fundamentals_annual.parquet | rows=7092 | bad_files=0


,cik,fy,Assets,Liabilities,CFO,NetIncomeLoss,Revenue,StockholdersEquity,CapEx,ticker_yahoo,security_name
0,0000001800,2009,52416623000,13049489000.0,NaN,NaN,30764707000.0,22855627000,NaN,ABT,Abbott Laboratories
1,0000001800,2010,59462266000,17262434000.0,NaN,NaN,9967800000.0,22388135000,NaN,ABT,Abbott Laboratories
2,0000001800,2011,60276893000,15480228000.0,NaN,NaN,10377400000.0,24439833000,NaN,ABT,Abbott Laboratories
3,0000001800,2012,67234944000,13280176000.0,NaN,NaN,10836900000.0,26720961000,NaN,ABT,Abbott Laboratories
4,0000001800,2013,42953000000,9507000000.0,3324000000.0,589000000.0,5655000000.0,25171000000,NaN,ABT,Abbott Laboratories
5,0000001800,2014,41275000000,10532000000.0,3675000000.0,905000000.0,5356000000.0,21526000000,NaN,ABT,Abbott Laboratories
6,0000001800,2015,41247000000,9186000000.0,2966000000.0,767000000.0,5188000000.0,21211000000,NaN,ABT,Abbott Laboratories
7,0000001800,2016,52666000000,6660000000.0,3203000000.0,798000000.0,5333000000.0,20538000000,NaN,ABT,Abbott Laboratories
8,0000001800,2017,76250000000,8912000000.0,5570000000.0,-828000000.0,7589000000.0,30897000000,NaN,ABT,Abbott Laboratories
9,0000001800,2018,67173000000,9012000000.0,6300000000.0,654000000.0,NaN,30524000000,NaN,ABT,Abbott Laboratories


""


In [12]:
fund_wide["Revenue"].isna().mean()
fund_wide["Liabilities"].isna().mean()


np.float64(0.048787366046249295)

In [1]:
# =========================================
# Cell 8) companyfacts -> fundamentals_quarterly (분기 데이터)
# =========================================
# 목적:
#   - SEC companyfacts에서 분기(Q1~Q4) 재무 데이터 추출
#   - 촉매 계산용 고빈도 데이터 확보
#
# 산출물:
#   - data/interim/fundamentals_quarterly.parquet
#
# 주의:
#   - fp in ["Q1","Q2","Q3","Q4"] 필터
#   - filed_date 포함 (look-ahead 방지용)

import gzip
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

OUT_FUND_QUARTERLY = INTERIM_DIR / "fundamentals_quarterly.parquet"

# ---- 표준 계정 정의 (Cell 7과 동일) ----
STD_MAP_Q = {
    "Assets": ["Assets"],
    "StockholdersEquity": [
        "StockholdersEquity",
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    ],
    "NetIncomeLoss": ["NetIncomeLoss", "ProfitLoss"],
    "CFO": ["NetCashProvidedByUsedInOperatingActivities"],
    "CapEx": ["CapitalExpenditures"],
    "Revenue": [
        "Revenues",
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "SalesRevenueGoodsNet",
        "SalesRevenueServicesNet",
        "RevenuesNetOfInterestExpense",
    ],
    "Liabilities": ["Liabilities"],
}

LIAB_FALLBACK_SUM_Q = ("LiabilitiesCurrent", "LiabilitiesNoncurrent")

_EMPTY_Q = pd.DataFrame(columns=["fy", "fp", "end", "filed", "unit", "value"])

def _get_quarterly_series(gaap: dict, concept: str) -> pd.DataFrame:
    """분기 데이터만 추출 (Q1, Q2, Q3, Q4)"""
    node = gaap.get(concept)
    if not node:
        return _EMPTY_Q.copy()

    rows = []
    units = node.get("units", {})
    for unit, items in units.items():
        if not isinstance(items, list):
            continue
        for it in items:
            fp = str(it.get("fp", "")).upper()
            # 분기만 필터 (Q1, Q2, Q3, Q4)
            if fp not in ["Q1", "Q2", "Q3", "Q4"]:
                continue
            fy = it.get("fy")
            end = it.get("end")
            filed = it.get("filed")
            val = it.get("val")
            if fy is None or end is None or val is None:
                continue
            rows.append({
                "fy": int(fy),
                "fp": fp,
                "end": pd.to_datetime(end, errors="coerce"),
                "filed": pd.to_datetime(filed, errors="coerce"),
                "unit": str(unit),
                "value": pd.to_numeric(val, errors="coerce"),
            })

    df = pd.DataFrame(rows) if rows else _EMPTY_Q.copy()
    df = df.dropna(subset=["fy", "end", "value"])
    if df.empty:
        return _EMPTY_Q.copy()

    # 같은 (fy, fp) 다중 관측치면 filed 최신값 우선
    df = df.sort_values(["fy", "fp", "filed"]).drop_duplicates(subset=["fy", "fp"], keep="last")
    return df

def _pick_best_unit_q(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return _EMPTY_Q.copy()
    df = df.copy()
    if df["unit"].str.contains("USD", case=False, na=False).any():
        df = df[df["unit"].str.contains("USD", case=False, na=False)].copy()
        df = df.sort_values(["fy", "fp", "filed"]).drop_duplicates(subset=["fy", "fp"], keep="last")
    return df if not df.empty else _EMPTY_Q.copy()

def _series_to_value_q(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=["fy", "fp", "end", "filed", name])
    out = df[["fy", "fp", "end", "filed", "value"]].rename(columns={"value": name}).copy()
    return out

def _outer_merge_on_fy_fp(base: pd.DataFrame, add: pd.DataFrame) -> pd.DataFrame:
    """(fy, fp) 기준 outer merge"""
    if base is None or base.empty:
        return add.copy()

    m = base.merge(add, on=["fy", "fp"], how="outer", suffixes=("", "_add"))

    for col in ["end", "filed"]:
        if col in m.columns and f"{col}_add" in m.columns:
            m[col] = pd.to_datetime(m[col], errors="coerce")
            m[f"{col}_add"] = pd.to_datetime(m[f"{col}_add"], errors="coerce")
            m[col] = m[[col, f"{col}_add"]].max(axis=1)
            m = m.drop(columns=[f"{col}_add"])

    return m

def _liabilities_fallback_q(gaap: dict) -> pd.DataFrame:
    """Liabilities가 없을 때 Current + Noncurrent 합산"""
    cur = _pick_best_unit_q(_get_quarterly_series(gaap, LIAB_FALLBACK_SUM_Q[0]))
    non = _pick_best_unit_q(_get_quarterly_series(gaap, LIAB_FALLBACK_SUM_Q[1]))

    if cur.empty and non.empty:
        return _EMPTY_Q.copy()

    cur_v = _series_to_value_q(cur, "cur")
    non_v = _series_to_value_q(non, "non")

    m = cur_v.merge(non_v, on=["fy", "fp"], how="outer", suffixes=("", "_x"))

    for col in ["end", "filed"]:
        if col not in m.columns:
            m[col] = pd.NaT
        if f"{col}_x" in m.columns:
            m[col] = pd.to_datetime(m[col], errors="coerce")
            m[f"{col}_x"] = pd.to_datetime(m[f"{col}_x"], errors="coerce")
            m[col] = m[[col, f"{col}_x"]].max(axis=1)
            m = m.drop(columns=[f"{col}_x"])

    m["cur"] = pd.to_numeric(m.get("cur"), errors="coerce").fillna(0.0)
    m["non"] = pd.to_numeric(m.get("non"), errors="coerce").fillna(0.0)
    m["value"] = m["cur"] + m["non"]
    m["unit"] = "USD"

    out = m[["fy", "fp", "end", "filed", "unit", "value"]].copy()
    out = out.dropna(subset=["fy", "fp"])
    out["fy"] = out["fy"].astype(int)
    out = out.sort_values(["fy", "fp", "filed"]).drop_duplicates(subset=["fy", "fp"], keep="last")
    return out if not out.empty else _EMPTY_Q.copy()

def extract_fundamentals_quarterly(obj: dict, cik: str) -> pd.DataFrame:
    """분기 재무제표 추출"""
    gaap = obj.get("facts", {}).get("us-gaap", {})

    series = {}
    for std_name, candidates in STD_MAP_Q.items():
        best = _EMPTY_Q.copy()
        for c in candidates:
            df = _pick_best_unit_q(_get_quarterly_series(gaap, c))
            if not df.empty:
                best = df
                break
        series[std_name] = best

    # Liabilities fallback
    if series["Liabilities"].empty:
        series["Liabilities"] = _liabilities_fallback_q(gaap)

    # wide 병합
    base = None
    for std_name in ["Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"]:
        df = series.get(std_name, _EMPTY_Q.copy())
        add = _series_to_value_q(df, std_name)
        base = _outer_merge_on_fy_fp(base, add)

    if base is None or base.empty:
        return pd.DataFrame(columns=["cik", "fy", "fp", "end", "filed", "Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"])

    base["cik"] = cik

    cols = ["cik", "fy", "fp", "end", "filed", "Assets", "Liabilities", "CFO", "NetIncomeLoss", "Revenue", "StockholdersEquity", "CapEx"]
    for c in cols:
        if c not in base.columns:
            base[c] = pd.NA

    base = base[cols].sort_values(["cik", "fy", "fp"]).reset_index(drop=True)
    return base

# ---- 실행 ----
paths = sorted((RAW_DIR / "sec" / "companyfacts").glob("*.json.gz"))
if len(paths) == 0:
    raise FileNotFoundError("No companyfacts .json.gz files found.")

logger.info("Extracting fundamentals quarterly: files=%d", len(paths))

bad_q = []
chunks_q = []
for p in tqdm(paths, total=len(paths)):
    cik = p.stem.replace(".json", "").zfill(10)
    try:
        obj = load_json_gz(p)
        chunks_q.append(extract_fundamentals_quarterly(obj, cik=cik))
    except Exception as e:
        bad_q.append({"file": p.name, "error": str(e)})
        logger.warning("Quarterly extract failed: %s | %s", p.name, e)

fund_q = pd.concat(chunks_q, ignore_index=True) if chunks_q else pd.DataFrame()
if fund_q.empty:
    raise RuntimeError("fundamentals_quarterly is empty.")

# ticker 매핑 추가
fund_q = fund_q.merge(
    master[["cik", "ticker_yahoo", "security_name"]].drop_duplicates(),
    on="cik",
    how="left",
)

save_parquet(fund_q, OUT_FUND_QUARTERLY)

logger.info("Saved fundamentals quarterly: %s | rows=%d | bad_files=%d", OUT_FUND_QUARTERLY, len(fund_q), len(bad_q))
print(f"\n✅ 분기 재무 데이터 저장 완료: {OUT_FUND_QUARTERLY}")
print(f"   rows: {len(fund_q):,}, tickers: {fund_q['ticker_yahoo'].nunique()}")
print(f"   기간: fy {fund_q['fy'].min()} ~ {fund_q['fy'].max()}")
print(f"   filed 범위: {fund_q['filed'].min().date()} ~ {fund_q['filed'].max().date()}")

display(fund_q.head(20))

NameError: name 'INTERIM_DIR' is not defined

In [ ]:
# =============================================================================
# [DATA PIPELINE MAP / QUICK REFERENCE]
# 이 노트북(01_bootstrap.ipynb)의 산출물 및 저장 경로 요약
# 다른 IPYNB 또는 새 대화창에서 빠르게 위치 확인용
# =============================================================================

# [PROJECT ROOT]
# QP2_ROOT = C:/QP2
# 모든 경로는 이 루트를 기준으로 상대 참조

# -------------------------------------------------------------------------
# [META / 기준 정보]
# -------------------------------------------------------------------------
# - data/meta/sec_ticker_cik.parquet
#   : SEC ticker → CIK 매핑 (companyfacts 접근용 SSOT)
#
# - data/meta/sp500_universe.parquet
#   : S&P500 기준 종목 마스터
#     컬럼: ticker_yahoo, cik, security_name, (GICS Sector 등)
#
# - data/meta/sp500_unmatched.parquet
#   : S&P500 중 CIK 매칭 실패 종목 (정상적이면 비어 있음)
#
# - data/meta/yahoo_failed.parquet
#   : Yahoo 증분 수집 중
#     - 신규 데이터 없음
#     - 다운로드 실패
#     - 병합 실패
#     등의 로그 기록용

# -------------------------------------------------------------------------
# [RAW DATA]
# -------------------------------------------------------------------------
# - data/raw/yahoo/{TICKER}.parquet
#   : Yahoo Finance 일봉 원자료
#     - Date 컬럼 기준
#     - 증분 업데이트 구조 (마지막 Date 이후만 추가)
#
# - data/raw/sec/companyfacts/{CIK}.json.gz
#   : SEC EDGAR companyfacts 원본
#     - us-gaap 전체 계정 포함
#     - 연간/분기 혼합 원자료

# -------------------------------------------------------------------------
# [INTERIM DATA / 정규화 결과]
# -------------------------------------------------------------------------
# - data/interim/yahoo_prices_long.parquet
#   : Yahoo 가격 패널 (long format)
#     컬럼: date, ticker, adj_close
#
# - data/interim/yahoo_adjclose_wide.parquet
#   : Yahoo 가격 패널 (wide format)
#     index: date
#     columns: ticker_yahoo
#
# - data/interim/fundamentals_annual.parquet
#   : 연간 재무제표 패널 (companyfacts 정규화 결과)
#     컬럼:
#       cik, fy,
#       Assets, Liabilities,
#       CFO, NetIncomeLoss,
#       Revenue, StockholdersEquity, CapEx,
#       ticker_yahoo, security_name
#
#   ⚠ 주의:
#   - fy 기준 연간 데이터
#   - filed/end 날짜는 현재 테이블에 포함되어 있지 않음
#   - 백테스트 시 look-ahead 방지용 filed_date 결합은 별도 셀에서 처리 예정

# -------------------------------------------------------------------------
# [증분 업데이트 동작 방식 요약]
# -------------------------------------------------------------------------
# Yahoo:
# - 각 ticker parquet의 마지막 Date를 읽음
# - NY 기준 오늘 이전까지만 요청
# - 신규 일자만 append
#
# SEC (companyfacts):
# - 현재는 "전체 재파싱" 구조
# - 증분을 하려면:
#   - etag / last_modified 비교
#   - filed_date 기준 신규 fy만 필터
#   구조를 추가하면 됨 (추후 확장)

# =============================================================================
# END OF PIPELINE MAP
# =============================================================================
